# PathSnap: data exploration and feature engineering
Run from the project root with the updated `pipeline.py` available as `pipeline.py` (rename the uploaded `pipeline(5).py` if needed). This notebook inventories the permitted `train` and `student_test` partitions, examines cleaning and candidate features, and exports derived weak labels. The `.route` file is **not** a GPS-point label and never enters candidate feature generation.

The raw source files are not modified. Invalid coordinates and duplicate timestamps are removed from cleaned observations; large time gaps and implausible speeds are retained and flagged.

In [2]:
# Project folders and shared functions.
from pathlib import Path
import pandas as pd
from pipeline_updated import (
    Config, FEATURE_NAMES, RoadIndex, align_reference,
    build_candidate_features, clean_track, filter_track_quality, discover_trips,
    inventory, make_map, process_dataset, read_graph, read_reference,
)

DATA = Path("Student_folder") / "Student folder"
RESULTS = Path("results")
RESULTS.mkdir(parents=True, exist_ok=True)
SETTINGS = Config()

assert DATA.is_dir(), f"Dataset folder not found: {DATA.resolve()}"
print("Dataset:", DATA.resolve())
print("Results:", RESULTS.resolve())

ModuleNotFoundError: No module named 'pyproj'

## Dataset inventory and quality checks
Inspect permitted trip counts, missing files, invalid GPS coordinates, duplicate timestamps, graph structure, reference-route continuity, and time/speed anomalies. The pipeline reports errors per trip; check them before proceeding.

In [ ]:
# Inventory all trips in the permitted development partitions.
trip_inventory = inventory(DATA)
trip_inventory.to_csv(RESULTS / "dataset_inventory.csv", index=False)
display(trip_inventory.groupby("partition").size().rename("trips"))
display(trip_inventory.groupby("partition")[[
    "raw_observations", "invalid_observations",
    "duplicate_timestamps_removed", "clean_observations",
    "gap_count", "speed_outliers"
]].sum(min_count=1))
errors = trip_inventory.loc[trip_inventory["status"] != "ok"]
if not errors.empty:
    display(errors[["partition", "trip_id", "status", "error"]])
assert errors.empty, "Fix dataset inventory errors before continuing"

## Preprocessing and features
`clean_track` first removes invalid coordinates/timestamps and duplicate timestamps. `filter_track_quality` then **removes** an observation if its incoming time gap exceeds 30 seconds, apparent speed exceeds 60 m/s, **or** it has no road candidate within 100 metres. These rules use motion between consecutive observations **before quality filtering**, avoiding cascading decisions. The first observation has no incoming step. Original `observation_id` values are retained.

`RoadIndex` projects coordinates to local UTM metres and retrieves nearby road arcs. The final candidate features cover geometry, heading, speed, timing, candidate rank, and node degrees. The reference `.route` is never read to construct prediction features.

In [ ]:
# Follow one training trip through quality filtering and feature creation.
trip = discover_trips(DATA, "train")[0]
gps, cleaning_report = clean_track(trip, SETTINGS)
nodes, arcs = read_graph(trip)
road = RoadIndex(nodes, arcs, SETTINGS)
gps, quality_report = filter_track_quality(gps, road, SETTINGS)
candidates, observations = build_candidate_features(gps, road, SETTINGS)

print("Initial cleaning audit:", cleaning_report)
print("Quality filter audit:", quality_report)
print("Retained GPS observations:", len(gps))
print("Remaining observations without candidates:", int((observations["candidate_count"] == 0).sum()))
print("Candidate feature rows:", len(candidates["X"]))
display(pd.DataFrame(candidates["X"], columns=FEATURE_NAMES).head())

## Reference alignment
Monotone alignment derives weak observation labels from the supplied arc sequence. Residual thresholds identify usable labels. These are not independently verified pointwise ground truth.

In [ ]:
# The route file gives ordered arcs, not a label for each GPS row.
# This step estimates rough labels and reports how many are usable.
route = read_reference(trip, len(arcs))
labels = align_reference(gps, road, route, SETTINGS)
print("GPS points with usable rough labels:", int(labels.label_usable.sum()))
display(labels.head())


## GPS and road map
The interactive Folium map shows **quality-filtered** GPS observations in blue, the supplied reference route in green, and an optional road-network layer in grey. It does not display model predictions.

In [ ]:
# Create the interactive Folium map using the updated pipeline.
from IPython.display import display
map_path = RESULTS / "maps" / f"{trip.name}.html"
html_map = make_map(trip, map_path)
print("Interactive map saved to:", map_path.resolve())
display(html_map)

## Export all permitted trips
`process_dataset` writes `candidates.npz`, `targets.npz`, `observations.csv.gz`, `labels.csv.gz`, and `metadata.json` per trip. It also writes `processing_summary.csv` and `manifest.json` under the processing output directory. Targets are derived weak labels; `fit_mask` excludes unreliable alignments and observations whose reference arc is absent from the candidate set. No evaluator partition is accessed. The processing summary also records counts removed for incoming gaps over 30 seconds, speeds over 60 m/s, and missing road candidates within 100 metres. Counts for the three criteria can overlap.

In [ ]:
# Generate candidate features and offline weak labels for permitted trips.
processed_root = RESULTS / "processed"
summary = process_dataset(DATA, processed_root, SETTINGS)
display(summary.groupby("partition")[[
    "raw_observations", "invalid_observations",
    "duplicate_timestamps_removed", "clean_observations",
    "candidate_rows", "no_candidate_observations",
    "usable_label_observations", "reference_candidate_misses", "fit_rows"
]].sum())
print("Summary:", (processed_root / "processing_summary.csv").resolve())
print("Manifest:", (processed_root / "manifest.json").resolve())

## Trip-level cross-validation preparation
The new shared pipeline does not implement `make_training_folds`. The following notebook-only step creates deterministic, grouped trip folds for **training trips only**. All rows from a trip must remain in the same validation fold. Fit any learned scaling only within each training fold.

In [ ]:
# Deterministic trip-level fold assignment; student_test is excluded.
from sklearn.model_selection import KFold

train_trips = discover_trips(DATA, "train")
n_splits = min(5, len(train_trips))
if n_splits < 2:
    raise ValueError("At least two training trips are required for cross-validation")
fold_rows = []
for fold_id, (_, validation_indices) in enumerate(
    KFold(n_splits=n_splits, shuffle=True, random_state=42).split(train_trips)
):
    fold_rows.extend({"trip_id": train_trips[i].name, "validation_fold": fold_id}
                     for i in validation_indices)
folds = pd.DataFrame(fold_rows).sort_values("trip_id").reset_index(drop=True)
folds.to_csv(RESULTS / "training_folds.csv", index=False)
display(folds.groupby("validation_fold").size().rename("trips"))

Next stages: train and compare five models, perform Bayesian tuning, then implement HMM/Viterbi, API and UI. Pointwise metrics must be described as agreement with derived weak labels.